# QRT-style ML mock 3: next-period volatility regression

**Core time:** 55-70 minutes

**Optional extension:** 20-30 minutes
**Goal:** predict a continuous next-five-day volatility target without temporal leakage.

## Scenario and rules

You receive chronological `train_data` and `test_data`. The target is `next_5d_volatility`.

- Preserve chronology and never shuffle.
- Fit all preprocessing inside a pipeline.
- Use walk-forward cross-validation with a five-row gap because adjacent five-day targets overlap.
- Use MAE as the primary selection metric; also report RMSE and R².
- Keep the labeled test period protected until model selection is complete.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 20261103
FEATURES = ["return_1d", "abs_return_1d", "volatility_5d", "volatility_20d", "volume_z", "cross_asset_corr_20d", "drawdown_60d"]
TARGET = "next_5d_volatility"
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
ROOT = Path.cwd()
if not (ROOT / "data" / "volatility_train.csv").exists():
    ROOT = ROOT / "QRT_mock_test"
train_data = pd.read_csv(ROOT / "data" / "volatility_train.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
test_data = pd.read_csv(ROOT / "data" / "volatility_test.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print("train:", train_data.shape, train_data["date"].min().date(), "to", train_data["date"].max().date())
print("protected test:", test_data.shape, test_data["date"].min().date(), "to", test_data["date"].max().date())
display(train_data.head())

def report_check(name, function):
    try:
        function()
        print(f"PASS: {name}")
    except NotImplementedError:
        print(f"NOT YET: {name}")
    except Exception as exc:
        print(f"CHECK: {name} -> {type(exc).__name__}: {exc}")

## Task 1 - Audit chronology and construct the validation period

Implement `chronological_train_validation`.

- Validate the required columns and monotonic dates.
- Sort by date defensively.
- Return the first 80% as `fitting` and the final 20% as `validation`.
- Do not shuffle and do not use any test rows.

In [ ]:
def chronological_train_validation(train_data: pd.DataFrame):
    # TODO: return fitting, validation in chronological order.
    raise NotImplementedError

fitting = None
validation = None
try:
    fitting, validation = chronological_train_validation(train_data)
    display(pd.DataFrame({"rows": [len(fitting), len(validation)], "start": [fitting["date"].min(), validation["date"].min()], "end": [fitting["date"].max(), validation["date"].max()]}, index=["fitting", "validation"]))
except NotImplementedError:
    print("Complete chronological_train_validation, then rerun this cell.")

In [ ]:
def check_chronological_split():
    fitting, validation = chronological_train_validation(train_data)
    assert len(fitting) == int(0.80 * len(train_data))
    assert len(fitting) + len(validation) == len(train_data)
    assert fitting["date"].is_monotonic_increasing
    assert validation["date"].is_monotonic_increasing
    assert fitting["date"].max() < validation["date"].min() < test_data["date"].min()

report_check("chronological split", check_chronological_split)

## Task 2 - Build the model and select Ridge alpha

Implement:

- `build_ridge_pipeline`: median imputation, standardization, then Ridge;
- `select_ridge_alpha`: `GridSearchCV` over `ALPHAS` using `TimeSeriesSplit(n_splits=5, gap=5)` and negative MAE.

The five-row gap prevents the overlapping five-day targets immediately before a validation fold from leaking information across the boundary.

In [ ]:
def build_ridge_pipeline(alpha: float = 1.0) -> Pipeline:
    # TODO: return SimpleImputer -> StandardScaler -> Ridge.
    raise NotImplementedError

def select_ridge_alpha(fitting: pd.DataFrame) -> GridSearchCV:
    # TODO: fit and return the requested time-series GridSearchCV.
    raise NotImplementedError

In [ ]:
def check_regression_model_selection():
    fitting, _ = chronological_train_validation(train_data)
    search = select_ridge_alpha(fitting)
    assert isinstance(search, GridSearchCV)
    assert isinstance(search.cv, TimeSeriesSplit)
    assert search.cv.gap == 5
    assert search.best_params_["model__alpha"] in ALPHAS
    assert [name for name, _ in search.best_estimator_.steps] == ["imputer", "scaler", "model"]

report_check("Ridge time-series model selection", check_regression_model_selection)

## Task 3 - Compare validation metrics with a baseline

Implement `regression_metrics` returning exactly `mae`, `rmse`, and `r2`.

Fit a mean `DummyRegressor` and the selected Ridge pipeline on `fitting`. Evaluate both on `validation`. Use MAE for the selection decision and explain what negative validation R² would mean.

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    # TODO: calculate MAE, RMSE, and R-squared.
    raise NotImplementedError

validation_results = None
ridge_search = None
try:
    fitting, validation = chronological_train_validation(train_data)
    dummy = DummyRegressor(strategy="mean").fit(fitting[FEATURES], fitting[TARGET])
    ridge_search = select_ridge_alpha(fitting)
    validation_results = pd.DataFrame(
        {
            "dummy": regression_metrics(validation[TARGET], dummy.predict(validation[FEATURES])),
            "ridge": regression_metrics(validation[TARGET], ridge_search.predict(validation[FEATURES])),
        }
    ).T
    display(validation_results)
    print("best alpha:", ridge_search.best_params_["model__alpha"])
except NotImplementedError:
    print("Complete Tasks 1-3, then rerun this cell.")

In [ ]:
def check_regression_metrics():
    fitting, validation = chronological_train_validation(train_data)
    search = select_ridge_alpha(fitting)
    metrics = regression_metrics(validation[TARGET], search.predict(validation[FEATURES]))
    assert set(metrics) == {"mae", "rmse", "r2"}
    assert all(np.isfinite(list(metrics.values())))
    assert metrics["mae"] > 0 and metrics["rmse"] >= metrics["mae"]
    assert metrics["r2"] > 0

report_check("regression metrics", check_regression_metrics)

## Task 4 - Final protected-test audit and coefficient extraction

After fixing alpha, refit Ridge on all training rows and inspect the test period once.

Implement:

- `make_prediction_audit`: `date`, `actual`, `predicted`, `residual`, `absolute_error`;
- `coefficient_table`: feature, standardized coefficient, and absolute coefficient sorted descending.

Compare final Ridge MAE with a mean baseline fitted on all training rows.

In [ ]:
def make_prediction_audit(model: Pipeline, test_data: pd.DataFrame) -> pd.DataFrame:
    # TODO: create the requested prediction and residual table.
    raise NotImplementedError

def coefficient_table(model: Pipeline) -> pd.DataFrame:
    # TODO: extract standardized Ridge coefficients from the fitted pipeline.
    raise NotImplementedError

test_audit = None
test_results = None
try:
    fitting, _ = chronological_train_validation(train_data)
    ridge_search = select_ridge_alpha(fitting)
    best_alpha = ridge_search.best_params_["model__alpha"]
    final_ridge = build_ridge_pipeline(best_alpha).fit(train_data[FEATURES], train_data[TARGET])
    final_dummy = DummyRegressor(strategy="mean").fit(train_data[FEATURES], train_data[TARGET])
    test_audit = make_prediction_audit(final_ridge, test_data)
    test_results = pd.DataFrame(
        {
            "dummy": regression_metrics(test_data[TARGET], final_dummy.predict(test_data[FEATURES])),
            "ridge": regression_metrics(test_audit["actual"], test_audit["predicted"]),
        }
    ).T
    display(test_results)
    display(coefficient_table(final_ridge))
    display(test_audit.head())
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    axes[0].plot(test_audit["date"], test_audit["actual"], label="actual")
    axes[0].plot(test_audit["date"], test_audit["predicted"], label="predicted")
    axes[0].set_ylabel("volatility")
    axes[0].legend()
    axes[1].plot(test_audit["date"], test_audit["residual"], color="purple")
    axes[1].axhline(0.0, color="black", linestyle="--")
    axes[1].set(ylabel="residual", xlabel="date")
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Complete Task 4 only after model selection is finished.")

In [ ]:
def check_final_regression_outputs():
    fitting, _ = chronological_train_validation(train_data)
    search = select_ridge_alpha(fitting)
    model = build_ridge_pipeline(search.best_params_["model__alpha"]).fit(train_data[FEATURES], train_data[TARGET])
    audit = make_prediction_audit(model, test_data)
    coefficients = coefficient_table(model)
    assert list(audit.columns) == ["date", "actual", "predicted", "residual", "absolute_error"]
    assert len(audit) == len(test_data)
    assert np.allclose(audit["absolute_error"], audit["residual"].abs())
    assert set(coefficients["feature"]) == set(FEATURES)

report_check("final regression outputs", check_final_regression_outputs)

## Optional advanced section

These tasks are not required for the core checks.

1. Fit a `RandomForestRegressor` on `fitting` and compare validation MAE with Ridge.
2. Calculate permutation importance on validation data using negative MAE and at least 10 repeats.
3. Explain why random cross-validation would be optimistic and how regime changes could damage both models.

In [ ]:
# OPTIONAL TODO: RandomForest validation and permutation importance.
advanced_regression_results = None
print("Optional section not run." if advanced_regression_results is None else advanced_regression_results)

## Final interpretation

In three or four sentences:

- quantify Ridge's improvement over the dummy baseline;
- interpret the largest standardized coefficients without claiming causality;
- describe any residual pattern visible through time;
- identify one production control you would add before using this forecast for risk sizing.

**Your answer:** TODO.